# Centinela QC: entrenar el detector en Colab (GPU) y exportarlo para el backend

El entrenamiento corre **solo acá**: el backend (Render) no entrena, carga el modelo ya entrenado. Este cuaderno entrena el detector YOLO, lo exporta a ONNX y deja listos los dos archivos que usa el backend:

- `centinela.onnx`: el detector, con la supresión de cajas (NMS) dentro del grafo.
- `centinela.json`: clases, tamaño de entrada, forma de la salida, umbral y bandas de llenado calibradas.

**Antes de empezar**, en la PC (con el backend corriendo para el etiquetado, ver `ml/README.md`):

```powershell
cd ml; .venv\Scripts\python autolabel.py; .venv\Scripts\python build_dataset.py
cd ..; Compress-Archive -Path ml\work\dataset -DestinationPath dataset.zip -Force
```

**En Colab:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*. Después, ejecutar las celdas en orden.

> Todo lo que está en `/content` se borra cuando se corta la sesión. Descargar `centinela_model.zip` apenas termine.

## 1. Comprobar la GPU

In [ ]:
!nvidia-smi

## 2. Subir `dataset.zip`

Ejecutar la celda y elegir el archivo. Si el zip ya está en Google Drive, usar la celda siguiente.

In [ ]:
from google.colab import files

uploaded = files.upload()  # elegir dataset.zip
assert "dataset.zip" in uploaded, f"Se esperaba dataset.zip y llegó: {list(uploaded)}"

In [ ]:
# Opcional: copiar el zip desde Google Drive en lugar de subirlo.
# from google.colab import drive
# drive.mount("/content/drive")
# !cp "/content/drive/MyDrive/dataset.zip" /content/

## 3. Descomprimir y revisar

In [ ]:
!rm -rf /content/dataset
!unzip -q -o /content/dataset.zip -d /content/

from pathlib import Path

ROOT = Path("/content/dataset")
assert (ROOT / "dataset.yaml").exists(), "No está /content/dataset/dataset.yaml: el zip tiene que contener la carpeta dataset/ en la raíz."
for split in ("train", "val", "test"):
    imgs = list((ROOT / "images" / split).glob("*.jpg"))
    lbls = list((ROOT / "labels" / split).glob("*.txt"))
    print(f"{split}: {len(imgs)} imágenes, {len(lbls)} etiquetas")

## 4. Corregir la ruta del `dataset.yaml`

`build_dataset.py` escribe la ruta de la PC (por ejemplo `C:/Users/...`). En Colab tiene que apuntar a `/content/dataset`.

In [ ]:
import yaml

DATA_YAML = ROOT / "dataset.yaml"
d = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
print("ruta original:", d["path"])
d["path"] = str(ROOT)
DATA_YAML.write_text(yaml.safe_dump(d, allow_unicode=True, sort_keys=False), encoding="utf-8")
print(DATA_YAML.read_text(encoding="utf-8"))

## 5. Instalar Ultralytics

In [ ]:
!pip install -q -U ultralytics
import ultralytics
ultralytics.checks()

## 6. Entrenar

Parámetros pensados para una botella de frente y pocas fotos (unas 100). `hsv_h` está bajo a propósito para que el modelo no confunda la tapa roja (Clásico) con la verde (Stevia).

In [ ]:
from ultralytics import YOLO

TRAIN = {
    "epochs": 60,
    "patience": 20,
    "imgsz": 640,
    "batch": 32,
    "workers": 2,
    "device": 0,        # GPU de Colab
    # Aumentaciones pensadas para una botella de frente: sin flip vertical, giro leve.
    "fliplr": 0.5,
    "flipud": 0.0,
    "degrees": 5.0,
    "translate": 0.08,
    "scale": 0.3,
    "hsv_h": 0.01,      # bajo para no confundir la tapa roja (Clásico) con la verde (Stevia)
    "hsv_s": 0.5,
    "hsv_v": 0.4,
    "mosaic": 0.5,
    "close_mosaic": 15,
}

model = YOLO("yolo26n.pt")  # nano, NMS-free; alternativa: yolo11n.pt
model.train(
    data=str(DATA_YAML),
    project="/content/runs",
    name="detect",
    exist_ok=True,
    plots=True,
    **TRAIN,
)

## 7. Ver resultados

In [ ]:
from IPython.display import Image, display

RUN = Path("/content/runs/detect")
for name in ("results.png", "confusion_matrix.png", "val_batch0_pred.jpg"):
    if (RUN / name).exists():
        print(name)
        display(Image(filename=str(RUN / name), width=900))

## 8. Exportar a ONNX para el backend

Mismo formato que espera `backend/src/lib/localModel.ts`: salida `[1, N, 6]` (`x1, y1, x2, y2, score, clase`), entrada letterbox de 640 px con relleno gris 114. Las bandas de llenado salen de `bands.json` (lo calibra `autolabel.py` con las fotos `ok` y `build_dataset.py` lo mete en el zip).

In [ ]:
import json, shutil
import numpy as np
!pip install -q onnx onnxslim onnxruntime
import onnxruntime as ort

BEST = RUN / "weights" / "best.pt"
assert BEST.exists(), "No está best.pt: revisar que el entrenamiento haya terminado."
CLASSES = ["botella", "tapa_ok", "tapa_mal_colocada", "tapa_abierta", "pico_sin_tapa", "menisco"]  # orden de ml/config.yaml
CONF = 0.25  # predict.conf en ml/config.yaml
IMGSZ = TRAIN["imgsz"]

# nms=True deja la supresión de cajas dentro del grafo: el backend recibe [1, N, 6] listo para usar.
onnx_path = YOLO(str(BEST)).export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False, nms=True)
OUT = Path("/content/centinela_model")
OUT.mkdir(exist_ok=True)
shutil.copyfile(onnx_path, OUT / "centinela.onnx")

sess = ort.InferenceSession(str(OUT / "centinela.onnx"), providers=["CPUExecutionProvider"])
inp = sess.get_inputs()[0]
out = sess.run(None, {inp.name: np.zeros((1, 3, IMGSZ, IMGSZ), dtype=np.float32)})[0]
assert out.ndim == 3 and out.shape[2] == 6, f"Salida inesperada {out.shape}: el backend necesita end2end [1, N, 6] (yolo26 o nms=True)."

bands_file = ROOT / "bands.json"
default = {"min": 0.22, "max": 0.34}
bands = json.loads(bands_file.read_text()) if bands_file.exists() else {"clasico": default, "stevia": default}
meta = {
    "classes": CLASSES,
    "imgsz": IMGSZ,
    "input": inp.name,
    "input_shape": inp.shape,
    "output_shape": list(out.shape),
    "output_layout": "end2end_xyxy_score_class",
    "letterbox_pad_value": 114,
    "conf": CONF,
    "bands": bands,
    "weights": BEST.name,
}
(OUT / "centinela.json").write_text(json.dumps(meta, ensure_ascii=False, indent=1), encoding="utf-8")
print(f"ONNX {(OUT / 'centinela.onnx').stat().st_size / 1e6:.1f} MB, salida {list(out.shape)}")
print(json.dumps(meta, ensure_ascii=False, indent=1))

## 9. Descargar el modelo

Descomprimir `centinela_model.zip` en `backend/models/` del repo (quedan `backend/models/centinela.onnx` y `backend/models/centinela.json`) y hacer commit y push: Render lo carga al desplegar. Otra opción, sin subir el ONNX al repo, es publicarlo (por ejemplo en un release de GitHub) y poner las URLs en `MODEL_URL` y `MODEL_META_URL` del servicio en Render.

In [ ]:
shutil.copyfile(BEST, OUT / "best.pt")  # por si hace falta reexportar
shutil.make_archive("/content/centinela_model", "zip", OUT)
files.download("/content/centinela_model.zip")

In [ ]:
# Opcional: guardar la corrida completa en Google Drive.
# from google.colab import drive
# drive.mount("/content/drive")
# !cp -r /content/runs "/content/drive/MyDrive/centinela_runs"